# DuckPD GPU Vector Search & Streaming Text Embeddings Walkthrough

This interactive notebook demonstrates explicit PyTorch GPU model preparation, bounded in-engine corpus embedding, persisted embedding metadata, and exact text search over remote and local Parquet datasets.

### Highlights
- **PyTorch GPU inference**: Run the pinned BGE model through `TransformersEmbeddingProvider` on NVIDIA CUDA or AMD ROCm.
- **Explicit accelerator selection**: Request `device="cuda"`; DuckPD raises instead of silently falling back to CPU.
- **Lazy remote streaming**: Scan Parquet directly over HTTPS without downloading the full source first.
- **Persisted model identity**: DuckPD writes a metadata sidecar with the Parquet vectors and restores it on read.
- **Sidecar-Inferred Search**: Omit `model=` after reload while retaining an optional explicit fingerprint assertion.


## 1. Select the ROCm kernel and configure the model

In VS Code, choose **Select Kernel** in the upper-right and select **DuckPD ROCm 7.2.4**. If it is absent, follow the environment and kernel-registration commands in `demo/generate_data/README.md`, then reload the VS Code window. The normal repository `.venv` intentionally does not include accelerator-specific packages.

The first code cell verifies the selected interpreter, imports both GPU dependencies, and confirms that PyTorch can see the GPU before model preparation. On AMD ROCm, PyTorch exposes the device through its `cuda` API.

The Transformers backend uses a distinct model fingerprint and output file, preventing its vectors from being mixed with prior FastEmbed CPU output.

In [ ]:
import importlib
import sys
from pathlib import Path
from time import perf_counter

import duckpd as pd

try:
    torch = importlib.import_module("torch")
    importlib.import_module("transformers")
except (ImportError, OSError) as error:
    raise RuntimeError(
        "This notebook is running the wrong VS Code kernel. "
        f"Current interpreter: {sys.executable}. Select Kernel > DuckPD ROCm 7.2.4, "
        "then restart the kernel. Setup instructions: demo/generate_data/README.md"
    ) from error
if not torch.cuda.is_available():
    raise RuntimeError(
        f"PyTorch in {sys.executable} cannot access a GPU. Select the DuckPD ROCm 7.2.4 "
        "kernel and restart it; DuckPD will not fall back to CPU."
    )

DATA_URL = "https://huggingface.co/datasets/AlphaDojo/dojo_stock_news/resolve/main/data.parquet"
DEMO_DIR = Path("demo") if Path("demo").is_dir() else Path("..")
ARTIFACT_DIR = DEMO_DIR / ".artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)
EMBEDDED_DATA = ARTIFACT_DIR / "nvidia-news-embedded-transformers.parquet"
QUERY = "AI chip demand and revenue growth"
TRANSFORMER_BATCH_SIZE = 64

MODEL = pd.embedding_model(
    "BAAI/bge-small-en-v1.5",
    revision="5c38ec7c405ec4b44b94cc5a9bb96e735b38267a",
    dimension=384,
    backend="transformers",
    pooling="cls",
)

print(f"Kernel interpreter: {sys.executable}")
print(f"GPU: {torch.cuda.get_device_name(0)} (ROCm {torch.version.hip})")
print(f"DuckPD version: {pd.__version__}")
print(
    f"Embedding model: {MODEL.model} "
    f"(revision: {MODEL.revision[:12]}..., dimension: {MODEL.dimension})"
)
print(f"Embedded dataset: {EMBEDDED_DATA}")

## 2. Prepare the model on the GPU

Register an explicit GPU provider before preparation. `device="cuda"` selects either NVIDIA CUDA or AMD ROCm according to the installed PyTorch build. Preparation verifies the immutable local model cache and reports the actual runtime.

In [ ]:
session = pd.connect()
provider = pd.TransformersEmbeddingProvider(
    MODEL,
    device="cuda",
    batch_size=TRANSFORMER_BATCH_SIZE,
)
session.register_embedding_provider(MODEL, provider)

preparation_started = perf_counter()
prepared = session.prepare_embedding_model(MODEL)
preparation_seconds = perf_counter() - preparation_started

print(f"Backend: {prepared.backend} via {prepared.execution_providers}")
print(f"Model preparation time: {preparation_seconds:.3f}s")
print(f"Persisted model fingerprint: {MODEL.fingerprint}")

## 3. Load or build the GPU-embedded dataset

If the Transformers-specific Parquet file exists locally, load it directly. Otherwise, DuckPD lazily scans the remote Hugging Face dataset, filters NVIDIA (`NVDA`) articles, computes embeddings on the registered GPU in bounded Arrow and model batches, and persists the result. The separate filename prevents accidental reuse of FastEmbed CPU vectors.

In [ ]:
if EMBEDDED_DATA.exists():
    embedded = session.read_parquet(EMBEDDED_DATA)
    dataset_status = f"Loaded {EMBEDDED_DATA}"
else:
    build_started = perf_counter()
    news = session.read_parquet(DATA_URL)
    nvidia_news = news[news["symbol"] == "NVDA"]
    embedded = nvidia_news.embed_text(
        columns=["title", "description"],
        into="embedding",
        model=MODEL,
        batch_size=64,
        null_policy="empty",
    )
    embedded.write_parquet(EMBEDDED_DATA)
    build_seconds = perf_counter() - build_started
    dataset_status = (
        f"Created {EMBEDDED_DATA} from the remote archive in {build_seconds:.3f} seconds"
    )
    embedded = session.read_parquet(EMBEDDED_DATA)

print(f"Embedding dataset: {dataset_status}")
print(f"Columns: {embedded.columns}")

## 4. Preview the source data

Inspect a few identifying columns with bounded materialization via `head()`. The large `embedding` vectors are intentionally omitted here; they remain available in the lazy frame for search.

In [ ]:
preview = embedded[["symbol", "title", "publisher", "publish_date"]].head(5)
preview

## 5. Run Sidecar-Inferred Vector Search

`write_parquet()` persisted the model identity beside the vector data, and `read_parquet()` restored it. The first search therefore omits `model=`. The second passes the model explicitly as a fingerprint compatibility assertion; it does not override the column metadata. Both plans must return identical results.


In [ ]:
query_started = perf_counter()
matches = embedded.vector.search_text(
    QUERY,
    column="embedding",
    metric="cosine",
    k=5,
    tie_breaker="title",
)[["symbol", "title", "publisher", "publish_date", "_distance"]]

explicit_matches = embedded.vector.search_text(
    QUERY,
    column="embedding",
    model=MODEL,
    metric="cosine",
    k=5,
    tie_breaker="title",
)[["symbol", "title", "publisher", "publish_date", "_distance"]]

logical_plan = matches.explain("json")
assert '"model_origin": "sidecar"' in logical_plan
assert QUERY not in logical_plan

result = matches.collect()
explicit_result = explicit_matches.collect()
assert result.equals(explicit_result)
query_seconds = perf_counter() - query_started

print(f"Query: {QUERY!r}")
print(f"Query-to-response for inferred and explicit plans: {query_seconds:.3f} seconds")
print("Sidecar-inferred and explicit-model results are identical.")

## 6. Inspect the Results

Lower cosine distance means greater semantic similarity. The successful parity assertion above proves that sidecar inference selected the same embedding space as the explicit model. Automatic preparation is catalog-scoped; this ordinary sidecar workflow prepared the model explicitly in Step 2.


In [ ]:
display(result)
session.close()